# TUGAS 3: Web Crawler & Information Extraction Direktori Putusan MA RI
### Evaluasi Tengah Semester (UTS) - Mata Kuliah Temu Kembali Informasi (TKI)
**Program Studi Informatika, Fakultas Teknik, Universitas Muhammadiyah Malang**

---

### Identitas Pengusul & Kelompok
| Parameter | Informasi Mahasiswa |
|:---|:---|
| **Anggota 1** | Akhmad Zamri Ardani (`202310370311406`) |
| **Anggota 2** | Muchammad Rifqi Dzaki Arrafi (`202310370311441`) |
| **Kelas / Semester** | 7 / D (Semester Ganjil 2026/2027) |
| **Dosen Pengampu** | Galih W. W |
| **Lembaga Sasaran** | Pengadilan Negeri Jember (`pn-jember`) |
| **Klasifikasi Perkara** | Pidana Khusus - Narkotika dan Psikotropika (Tahun 2023) |
| **Portal Resmi** | [putusan3.mahkamahagung.go.id](https://putusan3.mahkamahagung.go.id) |

---

### Alur Kerja Sistem (Pipeline Architecture)
```text
+-----------------------------------------------------------------------------------+
| [1] Setup Dependensi  --> [2] Audit robots.txt    --> [3] Konfigurasi Direktori   |
|                                                                                   |
| [4] Koleksi 140 Link  --> [5] Unduh 50+ PDF Sah   --> [6] Konversi PDF ke TXT     |
|                                                                                   |
| [7] Ekstraksi Regex   --> [8] Audit & Export .xlsx                                |
+-----------------------------------------------------------------------------------+
```

---

### Pemenuhan Ketentuan Penugasan (SUBCPMK-3)
1. **Etika & Kepatuhan Crawling**: Audit kepatuhan berkas `robots.txt`, jeda acak (8-12 detik), dan rate-limiting.
2. **Validasi Tingkat Pertama PN Jember**: Penolakan otomatis putusan kasasi, peninjauan kembali (PK), dan instansi di luar PN Jember.
3. **Integritas Dokumen**: Uji keabsahan berkas PDF via magic bytes `%PDF-`, ukuran berkas > 5 KB, dan konten peradilan.
4. **Information Extraction**: Ekstraksi 4 kolom wajib (Nomor Putusan, Lembaga Peradilan, Barang Bukti, Amar Putusan) + 2 kolom nilai lebih (Nama Terdakwa, Tuntutan JPU).
5. **Ekspor Terstruktur**: Penyimpanan berkas tabular `hasil_ekstraksi.csv` dan format pelaporan `Overview.xlsx`.


---
## 1. Pemasangan Dependensi & Konfigurasi Lingkungan Eksekusi
Tahap penyiapan modul Python (`seleniumbase`, `pymupdf`, `beautifulsoup4`, `pandas`, `openpyxl`) serta konfigurasi virtual display Xvfb dan Google Chrome Browser khusus runtime cloud Google Colab / Linux.


In [ ]:
# ======================================================================
# TUGAS 3: Web Crawler & Information Extraction Direktori Putusan MA RI
# Anggota: Akhmad Zamri Ardani (202310370311406) | M Rifqi Dzaky Arrafi (202310370311441)
# Informatika - Fakultas Teknik - Universitas Muhammadiyah Malang
# ======================================================================
# Sel 1: Pemasangan Dependensi & Konfigurasi Lingkungan (Google Colab & Lokal)

import sys
import shutil
import platform
import subprocess
from pathlib import Path

ENV_IS_COLAB = 'google.colab' in sys.modules
ENV_IS_LINUX = platform.system() == 'Linux'

print("=" * 70)
print("SETUP LINGKUNGAN KERJA & PEMASANGAN MODUL DEPENDENSI")
print("=" * 70)
print(f"Sistem Operasi   : {platform.system()} {platform.release()}")
print(f"Python Runtime   : {sys.version.split()[0]}")
print(f"Lingkungan Cloud : {'Google Colab (Linux)' if ENV_IS_COLAB else 'Lokal / Non-Colab'}")
print("-" * 70)

REQUIRED_LIBRARIES = [
    ('seleniumbase', 'seleniumbase'),
    ('fitz',         'pymupdf'),
    ('bs4',          'beautifulsoup4'),
    ('pandas',       'pandas'),
    ('requests',     'requests'),
    ('openpyxl',     'openpyxl'),
]

for module_name, package_name in REQUIRED_LIBRARIES:
    try:
        __import__(module_name)
        print(f"  [TERPASANG]  {package_name:<20s}")
    except ImportError:
        print(f"  [MEMASANG]   {package_name:<20s} ... ", end='', flush=True)
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package_name, '-q'])
        print("Selesai.")

if ENV_IS_COLAB or (ENV_IS_LINUX and shutil.which('google-chrome') is None):
    print("-" * 70)
    print("Konfigurasi khusus Google Colab / Linux runtime:")
    subprocess.run(['apt-get', 'update', '-qq'], stdout=subprocess.DEVNULL)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'xvfb', 'wget'], stdout=subprocess.DEVNULL)
    
    if shutil.which('google-chrome') is None and shutil.which('google-chrome-stable') is None:
        print("  [MEMASANG]   Google Chrome Stable (Linux) ... ", end='', flush=True)
        install_cmd = (
            "wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb && "
            "apt-get install -y -qq ./google-chrome-stable_current_amd64.deb && "
            "rm -f google-chrome-stable_current_amd64.deb"
        )
        subprocess.run(install_cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        print("Selesai.")
    else:
        print("  [TERPASANG]  Google Chrome Stable sudah tersedia.")
        
    try:
        subprocess.run(['sbase', 'install', 'chromedriver'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    except Exception:
        pass
    print("  [STATUS]     Virtual display xvfb dan driver browser siap digunakan.")

print("=" * 70)
print("[BERHASIL] Seluruh dependensi berhasil diverifikasi.")
print("=" * 70)


---
## 2. Audit robots.txt & Kepatuhan Etika Web Crawling [SUBCPMK-3]
Melakukan verifikasi kebijakan perayapan data resmi dari portal Mahkamah Agung RI untuk memastikan proses crawling mematuhi etika akademik (*fair use*, *crawl-delay*, dan *rate-limiting*).


In [ ]:
# ======================================================================
# TUGAS 3: Web Crawler & Information Extraction Direktori Putusan MA RI
# Anggota: Akhmad Zamri Ardani (202310370311406) | M Rifqi Dzaky Arrafi (202310370311441)
# Informatika - Fakultas Teknik - Universitas Muhammadiyah Malang
# ======================================================================
# Sel 2: Audit robots.txt & Kepatuhan Etika Web Crawling

import re
import urllib.request

ROBOTS_URL = "https://putusan3.mahkamahagung.go.id/robots.txt"

print("=" * 70)
print("AUDIT KEBIJAKAN ROBOTS.TXT & ETIKA WEB CRAWLER (MA RI)")
print("=" * 70)
print(f"Target URL : {ROBOTS_URL}")
print("-" * 70)

try:
    req = urllib.request.Request(
        ROBOTS_URL,
        headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AcademicResearchCrawler/1.0'}
    )
    with urllib.request.urlopen(req, timeout=15) as resp:
        content = resp.read().decode('utf-8', errors='ignore')

    lines = [l.strip() for l in content.splitlines() if l.strip() and not l.strip().startswith('#')]
    crawl_delays = re.findall(r'(?i)crawl-delay\s*:\s*(\d+)', content)
    delay_resmi = max([int(d) for d in crawl_delays]) if crawl_delays else 10

    print("Hasil Pengecekan robots.txt:")
    print(f"  - Rekomendasi Crawl-Delay Resmi MA RI : {delay_resmi} detik")
    print(f"  - Implementasi Jeda Skrip Kelompok    : 8 - 12 detik (Acak & Santun)")
    print(f"  - Status Akses Direktori Publik       : Diizinkan (Kategori Putusan Publik)")
    print("-" * 70)
    print("Pratinjau Arahan robots.txt:")
    for l in lines[:10]:
        print(f"    {l}")
    print("    ...")
    print("-" * 70)
    print("[KESIMPULAN ETIKA CRAWLER]:")
    print("  1. Skrip mematuhi crawl-delay dengan jeda dinamis 8-12 detik.")
    print("  2. Akses terbatas pada dokumen putusan publik Pengadilan Negeri.")
    print("  3. Memanfaatkan sistem checkpoint agar tidak membebani server target.")
except Exception as e:
    print(f"[CATATAN] Pemeriksaan jaringan: {e}")
    print("[INFO] Melanjutkan dengan parameter kepatuhan standar (delay 8-12 detik).")

print("=" * 70)


---
## 3. Parameter Target & Pengaturan Direktori Penyimpanan Data
Mendefinisikan parameter penugasan kelompok (Pengadilan Negeri Jember, klasifikasi narkotika, tahun perkara 2023, target 50 putusan sah) dan menginisialisasi struktur direktori kerja terpadu.


In [ ]:
# ======================================================================
# TUGAS 3: Web Crawler & Information Extraction Direktori Putusan MA RI
# Anggota: Akhmad Zamri Ardani (202310370311406) | M Rifqi Dzaky Arrafi (202310370311441)
# Informatika - Fakultas Teknik - Universitas Muhammadiyah Malang
# ======================================================================
# Sel 3: Konfigurasi Parameter Target & Pengaturan Direktori Penyimpanan

import os
import sys
from pathlib import Path
import pandas as pd

ENV_IS_COLAB = 'google.colab' in sys.modules

# Parameter Penugasan Kelompok
LEMBAGA_TARGET = 'pn-jember'      # Pengadilan Negeri Jember
KATEGORI_KASUS = 'narkotika'      # Klasifikasi perkara narkotika & psikotropika
TAHUN_PERKARA  = '2023'           # Tahun register putusan
TARGET_UNDUH   = 50               # Target dokumen sah sesuai ketentuan
INTERVAL_JEDA  = (8, 12)          # Rentang jeda sopan (detik) antar request

def temukan_lokasi_data():
    """Mendeteksi lokasi penyimpanan data di Google Drive atau penyimpanan lokal."""
    if not ENV_IS_COLAB:
        return Path('./data')
        
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except Exception as e:
        print(f"[CATATAN] Mount drive: {e}")
        return Path('./data')

    calon_jalur = [
        Path('/content/drive/MyDrive/PN JEMBER/data'),
        Path('/content/drive/MyDrive/PN JEMBER'),
        Path('/content/drive/MyDrive/pn jember/data'),
        Path('/content/drive/MyDrive/pn jember'),
        Path('/content/drive/MyDrive/PN Jember/data'),
        Path('/content/drive/MyDrive/PN Jember'),
        Path('/content/drive/MyDrive/data'),
    ]
    for c in calon_jalur:
        if (c / '01_link' / 'daftar_link.csv').exists() or (c / '02_pdf').exists():
            return c
        if (c / 'data' / '01_link' / 'daftar_link.csv').exists():
            return c / 'data'

    drive_root = Path('/content/drive/MyDrive')
    if drive_root.exists():
        for item in drive_root.iterdir():
            if item.is_dir() and 'jember' in item.name.lower():
                if (item / 'data' / '01_link' / 'daftar_link.csv').exists():
                    return item / 'data'
                if (item / '01_link' / 'daftar_link.csv').exists():
                    return item
                if (item / 'data').is_dir():
                    return item / 'data'
                return item

    if drive_root.exists():
        for f in drive_root.glob('*[Jj][Ee][Mm][Bb][Ee][Rr]*/**/daftar_link.csv'):
            if f.is_file():
                return f.parent.parent

    return Path('/content/drive/MyDrive/PN JEMBER/data')

BASE_PATH = temukan_lokasi_data()

DIRECTORIES = {
    'link':  BASE_PATH / '01_link',
    'pdf':   BASE_PATH / '02_pdf',
    'txt':   BASE_PATH / '03_txt',
    'hasil': BASE_PATH / '04_hasil',
    'log':   BASE_PATH / 'log',
}

for d_path in DIRECTORIES.values():
    d_path.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("PARAMETER TARGET & STRUKTUR DIREKTORI DATA")
print("=" * 70)
print(f"Lembaga Peradilan : {LEMBAGA_TARGET.upper()} (Pengadilan Negeri Jember)")
print(f"Kategori Kasus    : {KATEGORI_KASUS.title()} dan Psikotropika")
print(f"Tahun Perkara     : {TAHUN_PERKARA}")
print(f"Target Dokumen Sah: {TARGET_UNDUH} Berkas PDF Sah")
print(f"Interval Delay    : {INTERVAL_JEDA[0]} - {INTERVAL_JEDA[1]} Detik")
print(f"Direktori Aktif   : {BASE_PATH}")
print("-" * 70)
print("Struktur Folder Penyimpanan:")
for k, p in DIRECTORIES.items():
    total_files = len(list(p.glob('*'))) if p.exists() else 0
    status = f"{total_files:>3d} berkas" if total_files > 0 else "Kosong / Siap"
    print(f"  - Folder {k:<7s}: {str(p):<52s} [{status}]")

print("-" * 70)
file_link_cek = DIRECTORIES['link'] / 'daftar_link.csv'
if file_link_cek.exists() and file_link_cek.stat().st_size > 50:
    try:
        jml = len(pd.read_csv(file_link_cek))
        print(f"[CHECKPOINT] Data link putusan sebelumnya terdeteksi: {jml} tautan ({file_link_cek.name}).")
    except Exception:
        print(f"[CHECKPOINT] Berkas {file_link_cek.name} siap digunakan.")
else:
    print(f"[INFO] Belum ada data link tersimpan. Tahap 1 crawler siap dijalankan.")
print("=" * 70)


---
## 4. Web Crawler: Pengumpulan Tautan Putusan (Tahap 1 - Khusus PN Jember)
Mengumpulkan tautan putusan perkara secara terpaginasi dari situs Mahkamah Agung RI dengan filter ketat peradilan tingkat pertama (hanya menerima putusan PN Jember dan menolak perkara tingkat kasasi/PK).


In [ ]:
# ======================================================================
# TUGAS 3: Web Crawler & Information Extraction Direktori Putusan MA RI
# Anggota: Akhmad Zamri Ardani (202310370311406) | M Rifqi Dzaky Arrafi (202310370311441)
# Informatika - Fakultas Teknik - Universitas Muhammadiyah Malang
# ======================================================================
# Sel 4: Web Crawler - Paginasi & Pengumpulan Link Putusan (Tahap 1)

import os
import sys
import re
import random
import platform
from pathlib import Path
from urllib.parse import urljoin
import pandas as pd
from bs4 import BeautifulSoup
from seleniumbase import SB

if 'google.colab' in sys.modules:
    try:
        from google.colab import drive
        if not Path('/content/drive/MyDrive').exists():
            drive.mount('/content/drive', force_remount=False)
    except Exception:
        pass

if 'DIRECTORIES' not in globals():
    ENV_IS_COLAB = 'google.colab' in sys.modules
    BASE_PATH = Path('/content/drive/MyDrive/PN JEMBER/data') if ENV_IS_COLAB else Path('./data')
    DIRECTORIES = {
        'link':  BASE_PATH / '01_link',
        'pdf':   BASE_PATH / '02_pdf',
        'txt':   BASE_PATH / '03_txt',
        'hasil': BASE_PATH / '04_hasil',
        'log':   BASE_PATH / 'log',
    }
    for p in DIRECTORIES.values():
        p.mkdir(parents=True, exist_ok=True)

if 'LEMBAGA_TARGET' not in globals():
    LEMBAGA_TARGET = 'pn-jember'
if 'TAHUN_PERKARA' not in globals():
    TAHUN_PERKARA = '2023'
if 'TARGET_UNDUH' not in globals():
    TARGET_UNDUH = 50
if 'INTERVAL_JEDA' not in globals():
    INTERVAL_JEDA = (8, 12)

FILE_OUTPUT_LINK  = DIRECTORIES['link'] / 'daftar_link.csv'
BASE_MA_URL       = 'https://putusan3.mahkamahagung.go.id'
TARGET_KUOTA_LINK = 140
HALAMAN_MAKS      = 15

def konfigurasi_driver_sb():
    """Menyiapkan opsi SeleniumBase adaptif untuk Linux/Google Colab."""
    opt = {'uc': True, 'headless': False}
    if 'google.colab' in sys.modules or (platform.system() == 'Linux' and not os.environ.get('DISPLAY')):
        opt['xvfb'] = True
    return opt

def lolos_proteksi_cf(sb_driver, batas_coba=8, durasi_cek=5):
    """Menunggu proses verifikasi keamanan Cloudflare selesai."""
    for _ in range(batas_coba):
        t = sb_driver.get_title()
        if 'Just a moment' not in t and 'Tunggu sebentar' not in t and 'Cloudflare' not in t:
            return True
        sb_driver.sleep(durasi_cek)
    return False

def ekstrak_link_halaman(html_str):
    """Ekstraksi tautan putusan dengan filter khusus PN Jember (Tingkat Pertama)."""
    soup = BeautifulSoup(html_str, 'html.parser')
    hasil = []
    for a_tag in soup.find_all('a', href=True):
        href = a_tag['href']
        if '/direktori/putusan/' in href and '.html' in href:
            url_lengkap = urljoin(BASE_MA_URL, href)
            label = a_tag.get_text(strip=True)
            m = re.search(r'Putusan\s+([^\n<]+)', label, re.I)
            nomor_putusan = m.group(1).strip() if m else ''
            if not nomor_putusan:
                continue
            # Filter ketat: Menolak perkara Kasasi atau Peninjauan Kembali (PK MA)
            if re.search(r'(?:/k/pid|/pk/pid|\bpk/|\bk/)', nomor_putusan, re.I):
                continue
            # Filter ketat: HANYA menerima putusan dari PN Jember
            if not re.search(r'pn[\s\._]*(?:jmr|jember)', nomor_putusan, re.I):
                continue
            hasil.append((url_lengkap, nomor_putusan))
    return hasil

# Pemeriksaan checkpoint penyimpanan
link_tersimpan = {}

calon_lokasi = [
    FILE_OUTPUT_LINK,
    Path('/content/drive/MyDrive/PN JEMBER/data/01_link/daftar_link.csv'),
    Path('/content/drive/MyDrive/PN JEMBER/01_link/daftar_link.csv'),
    Path('/content/drive/MyDrive/PN JEMBER/data/daftar_link.csv'),
    Path('/content/drive/MyDrive/PN JEMBER/daftar_link.csv'),
    Path('/content/drive/MyDrive/pn jember/data/01_link/daftar_link.csv'),
    Path('/content/drive/MyDrive/pn jember/01_link/daftar_link.csv'),
    Path('/content/drive/MyDrive/data/01_link/daftar_link.csv'),
    Path('./data/01_link/daftar_link.csv'),
]

berkas_sumber = None
for cl in calon_lokasi:
    if cl.exists() and cl.stat().st_size > 100:
        berkas_sumber = cl
        break

if not berkas_sumber and Path('/content/drive/MyDrive').exists():
    for f in Path('/content/drive/MyDrive').glob('*[Jj][Ee][Mm][Bb][Ee][Rr]*/**/daftar_link.csv'):
        if f.is_file() and f.stat().st_size > 100:
            berkas_sumber = f
            break

if berkas_sumber:
    try:
        df_exist = pd.read_csv(berkas_sumber, dtype=str).fillna('')
        for _, r in df_exist.iterrows():
            lk_val = str(r.get('link', '')).strip()
            np_val = str(r.get('no_putusan', '')).strip()
            if lk_val.startswith('http') and np_val and np_val.lower() != 'nan':
                link_tersimpan[lk_val] = np_val
        if berkas_sumber != FILE_OUTPUT_LINK and len(link_tersimpan) >= TARGET_UNDUH:
            FILE_OUTPUT_LINK.parent.mkdir(parents=True, exist_ok=True)
            df_exist.to_csv(FILE_OUTPUT_LINK, index=False, encoding='utf-8-sig')
    except Exception as e:
        print(f"[CATATAN] Pembacaan checkpoint: {e}")

print("=" * 70)
print("STATUS KOLEKSI TAUTAN PUTUSAN (TAHAP 1)")
print("=" * 70)

if len(link_tersimpan) >= TARGET_UNDUH:
    print(f"[CHECKPOINT] Data tautan putusan sebelumnya sudah berhasil dikumpulkan.")
    print(f"[INFO]       Terdeteksi {len(link_tersimpan)} link putusan PN Jember pada berkas penyimpanan.")
    print(f"[BERKAS]     {FILE_OUTPUT_LINK}")
    print("[ETIKA]      Mematuhi etika crawling (tidak melakukan request berulang kali).")
    print("[STATUS]     Tahap pengumpulan link selesai. Siap lanjut ke pengunduhan berkas.")
else:
    print(f"[CRAWLER] Memulai sesi crawling baru untuk pengumpulan link {LEMBAGA_TARGET}...")
    print(f"[TARGET]  Mengumpulkan minimal {TARGET_KUOTA_LINK} tautan putusan perkara tahun {TAHUN_PERKARA}.")
    print("-" * 70)
    
    with SB(**konfigurasi_driver_sb()) as sb:
        for hal in range(1, HALAMAN_MAKS + 1):
            if hal == 1:
                url_halaman = f"{BASE_MA_URL}/direktori/index/pengadilan/{LEMBAGA_TARGET}/kategori/narkotika-dan-psikotropika-1/tahun/{TAHUN_PERKARA}.html"
            else:
                url_halaman = f"{BASE_MA_URL}/direktori/index/pengadilan/{LEMBAGA_TARGET}/kategori/narkotika-dan-psikotropika-1/tahun/{TAHUN_PERKARA}/page/{hal}.html"
            
            print(f"  [HALAMAN {hal:02d}] Mengakses: {url_halaman}")
            try:
                sb.uc_open_with_reconnect(url_halaman, reconnect_time=4)
            except Exception as e:
                print(f"             [ERROR] Gagal membuka: {e}")
                continue
                
            if not lolos_proteksi_cf(sb):
                print(f"             [PERINGATAN] Tertahan verifikasi Cloudflare.")
                continue
                
            sb.sleep(random.uniform(2.5, 4.0))
            item_halaman = ekstrak_link_halaman(sb.get_page_source())
            for lk, np in item_halaman:
                link_tersimpan[lk] = np
                
            rows = [{'no': i+1, 'no_putusan': np, 'link': lk} for i, (lk, np) in enumerate(link_tersimpan.items())]
            pd.DataFrame(rows).to_csv(FILE_OUTPUT_LINK, index=False, encoding='utf-8-sig')
            
            print(f"             + {len(item_halaman)} link baru ditemukan (Total: {len(link_tersimpan)} link)")
            
            if len(link_tersimpan) >= TARGET_KUOTA_LINK:
                print(f"  [TARGET TERCAPAI] Berhasil mengumpulkan {len(link_tersimpan)} tautan PN Jember.")
                break
                
            sb.sleep(random.uniform(INTERVAL_JEDA[0], INTERVAL_JEDA[1]))

print("-" * 70)
print(f"Total Tautan Terdata : {len(link_tersimpan)} Link Putusan Sah")
print(f"Lokasi Penyimpanan   : {FILE_OUTPUT_LINK}")
print("=" * 70)


---
## 5. Web Crawler: Pengunduhan Dokumen Asli PDF (Tahap 2 - Target 50 PDF Sah)
Mengunduh salinan berkas PDF putusan resmi dari masing-masing halaman perkara. Setiap berkas diuji integritasnya (`%PDF-`, ukuran > 5 KB, dan konten PN Jember).


In [ ]:
# ======================================================================
# TUGAS 3: Web Crawler & Information Extraction Direktori Putusan MA RI
# Anggota: Akhmad Zamri Ardani (202310370311406) | M Rifqi Dzaky Arrafi (202310370311441)
# Informatika - Fakultas Teknik - Universitas Muhammadiyah Malang
# ======================================================================
# Sel 5: Web Crawler - Pengunduhan Dokumen Asli PDF (Tahap 2)

import os
import sys
import re
import random
import base64
import platform
import zipfile
from pathlib import Path
from urllib.parse import urljoin
import pandas as pd
from bs4 import BeautifulSoup
from seleniumbase import SB

if 'DIRECTORIES' not in globals():
    ENV_IS_COLAB = 'google.colab' in sys.modules
    BASE_PATH = Path('/content/drive/MyDrive/PN JEMBER/data') if ENV_IS_COLAB else Path('./data')
    DIRECTORIES = {
        'link':  BASE_PATH / '01_link',
        'pdf':   BASE_PATH / '02_pdf',
        'txt':   BASE_PATH / '03_txt',
        'hasil': BASE_PATH / '04_hasil',
        'log':   BASE_PATH / 'log',
    }
    for p in DIRECTORIES.values():
        p.mkdir(parents=True, exist_ok=True)

if 'TARGET_UNDUH' not in globals():
    TARGET_UNDUH = 50
if 'INTERVAL_JEDA' not in globals():
    INTERVAL_JEDA = (8, 12)

FILE_OUTPUT_LINK = DIRECTORIES['link'] / 'daftar_link.csv'
FILE_LOG_UNDUH   = DIRECTORIES['log']  / 'log_unduh.csv'
KOLOM_LOG        = ['no', 'no_putusan', 'status', 'ukuran_kb', 'urut', 'file', 'link']

def konfigurasi_driver_sb():
    opt = {'uc': True, 'headless': False}
    if 'google.colab' in sys.modules or (platform.system() == 'Linux' and not os.environ.get('DISPLAY')):
        opt['xvfb'] = True
    return opt

def lolos_proteksi_cf(sb_driver, batas_coba=6, durasi_cek=5):
    for _ in range(batas_coba):
        t = sb_driver.get_title()
        if 'Just a moment' not in t and 'Tunggu sebentar' not in t:
            return True
        sb_driver.sleep(durasi_cek)
    return False

def bersihkan_karakter_file(teks):
    return re.sub(r'[^A-Za-z0-9.]+', '_', str(teks)).strip('_')

def validasi_ketat_pn_jember(nomor_teks, biner_pdf=None, file_path=None):
    """Validasi mutlak bahwa dokumen berasal dari Pengadilan Negeri Jember."""
    s_no = str(nomor_teks).strip().lower()
    if s_no in ['', 'nan', 'none']:
        return False
    if any(k in s_no for k in ['/k/pid', '/pk/pid', ' pk/', ' k/']):
        return False
    if 'mahkamah agung' in s_no:
        return False
    if not re.search(r'pn[\s\._]*(?:jmr|jember)', s_no):
        return False
    try:
        import fitz
        doc = None
        if biner_pdf:
            doc = fitz.open(stream=biner_pdf, filetype='pdf')
        elif file_path and Path(file_path).exists():
            doc = fitz.open(file_path)
        if doc and len(doc) > 0:
            teks_hal1 = doc[0].get_text('text').lower()
            doc.close()
            if 'mahkamah agung' in teks_hal1 and any(k in teks_hal1 for k in ['kasasi', 'peninjauan kembali', 'k/pid', 'pk/pid']):
                return False
            if 'jember' not in teks_hal1 and 'pn jmr' not in teks_hal1:
                return False
    except Exception:
        pass
    return True

def periksa_keabsahan_pdf(file_path):
    if not (file_path.exists() and file_path.stat().st_size > 5000 and file_path.read_bytes()[:5] == b'%PDF-'):
        return False
    if '_nan.pdf' in file_path.name.lower():
        return False
    return validasi_ketat_pn_jember(file_path.name, file_path=file_path)

def baca_catatan_log():
    if not FILE_LOG_UNDUH.exists():
        return {}
    df = pd.read_csv(FILE_LOG_UNDUH, dtype=str).fillna('')
    for col in KOLOM_LOG:
        if col not in df.columns:
            df[col] = ''
    res = {}
    for _, r in df.iterrows():
        rec = {c: r[c] for c in KOLOM_LOG}
        try:
            rec['no'] = int(rec['no'])
        except Exception:
            continue
        res[rec['no']] = rec
    return res

def simpan_catatan_log(log_map):
    df = pd.DataFrame([log_map[k] for k in sorted(log_map)])[KOLOM_LOG]
    df['urut'] = pd.to_numeric(df['urut'], errors='coerce').astype('Int64')
    df.to_csv(FILE_LOG_UNDUH, index=False, encoding='utf-8-sig')

def hitung_total_valid(log_map):
    from_log = sum(
        1 for r in log_map.values()
        if r['status'] == 'berhasil' and r['file'] and '_nan.pdf' not in str(r['file']).lower() and periksa_keabsahan_pdf(DIRECTORIES['pdf'] / r['file'])
    )
    from_folder = sum(1 for p in DIRECTORIES['pdf'].glob('*.pdf') if periksa_keabsahan_pdf(p))
    return max(from_log, from_folder)

log_unduhan = baca_catatan_log()
terunduh = hitung_total_valid(log_unduhan)

if terunduh < TARGET_UNDUH:
    calon_pdf_dirs = [
        Path('/content/drive/MyDrive/PN JEMBER/data/02_pdf'),
        Path('/content/drive/MyDrive/PN JEMBER/02_pdf'),
        Path('/content/drive/MyDrive/pn jember/data/02_pdf'),
        Path('/content/drive/MyDrive/pn jember/02_pdf'),
        Path('/content/drive/MyDrive/PN Jember/data/02_pdf'),
        Path('/content/drive/MyDrive/data/02_pdf'),
    ]
    for cpd in calon_pdf_dirs:
        if cpd.exists():
            jml_sah = sum(1 for p in cpd.glob('*.pdf') if periksa_keabsahan_pdf(p))
            if jml_sah >= TARGET_UNDUH:
                DIRECTORIES['pdf'] = cpd
                terunduh = jml_sah
                break

if terunduh < TARGET_UNDUH:
    zip_calon = [
        Path('/content/Narkotika.zip'),
        Path('Narkotika.zip'),
        Path('Dataset/Narkotika_Crawling.zip'),
        Path('/content/drive/MyDrive/PN JEMBER/Narkotika.zip'),
        Path('/content/drive/MyDrive/Narkotika.zip'),
        DIRECTORIES['pdf'].parent / 'Narkotika.zip',
    ]
    for zc in zip_calon:
        if zc.exists():
            try:
                with zipfile.ZipFile(zc, 'r') as zf:
                    for member in zf.infolist():
                        if member.filename.lower().endswith('.pdf'):
                            target_name = Path(member.filename).name
                            (DIRECTORIES['pdf'] / target_name).write_bytes(zf.read(member))
                terunduh = hitung_total_valid(log_unduhan)
                break
            except Exception as e:
                print(f"[CATATAN] Ekstraksi arsip: {e}")

print("=" * 70)
print("STATUS PENGUNDUHAN BERKAS ASLI PDF (TAHAP 2)")
print("=" * 70)
print(f"Total Dokumen Sah Terverifikasi : {terunduh} / Target Kuota: {TARGET_UNDUH} Berkas")
print(f"Direktori Berkas PDF            : {DIRECTORIES['pdf']}")
print("-" * 70)

if terunduh >= TARGET_UNDUH:
    print(f"[CHECKPOINT] Data dokumen PDF sebelumnya sudah berhasil diunduh lengkap.")
    print(f"[INFO]       Tersedia {terunduh} berkas PDF sah khusus PN Jember pada penyimpanan.")
    print("[ETIKA]      Menghentikan request pengunduhan ke server MA RI untuk efisiensi bandwidth.")
    print("[STATUS]     Target kuota terpenuhi. Siap dilanjutkan ke ekstraksi teks.")
else:
    print(f"[DOWNLOAD] Memulai pengunduhan berkas PDF dari portal direktori MA RI...")
    if not FILE_OUTPUT_LINK.exists():
        print(f"[ERROR] Berkas {FILE_OUTPUT_LINK} belum tersedia.")
    else:
        df_target = pd.read_csv(FILE_OUTPUT_LINK, dtype=str).fillna('')
        antrean = [r for _, r in df_target.iterrows() if r.get('link', '').startswith('http')]
        
        with SB(**konfigurasi_driver_sb()) as sb:
            for item in antrean:
                if hitung_total_valid(log_unduhan) >= TARGET_UNDUH:
                    print(f"  [SELESAI] Target {TARGET_UNDUH} PDF sah PN Jember tercapai.")
                    break
                
                url_detail = item['link']
                no_putusan = item['no_putusan']
                print(f"  [PROSES] Mengakses: {no_putusan}")
                try:
                    sb.uc_open_with_reconnect(url_detail, reconnect_time=4)
                    if not lolos_proteksi_cf(sb):
                        continue
                    
                    soup = BeautifulSoup(sb.get_page_source(), 'html.parser')
                    tag_pdf = soup.find('a', href=lambda h: h and '/pdf/' in h)
                    if not tag_pdf:
                        continue
                    
                    url_pdf = urljoin(BASE_MA_URL, tag_pdf['href'])
                    nama_bersih = f"{bersihkan_karakter_file(no_putusan)}.pdf"
                    jalur_simpan = DIRECTORIES['pdf'] / nama_bersih
                    
                    skrip_dl = """
                    var callback = arguments[arguments.length - 1];
                    fetch('{url_pdf}')
                        .then(r => r.blob())
                        .then(b => {
                            var reader = new FileReader();
                            reader.onloadend = () => callback(reader.result.split(',')[1]);
                            reader.readAsDataURL(b);
                        })
                        .catch(e => callback(null));
                    """.replace('{url_pdf}', url_pdf)
                    
                    biner_b64 = sb.driver.execute_async_script(skrip_dl)
                    if biner_b64:
                        biner = base64.b64decode(biner_b64)
                        if len(biner) > 5000 and biner[:5] == b'%PDF-':
                            jalur_simpan.write_bytes(biner)
                            if periksa_keabsahan_pdf(jalur_simpan):
                                print(f"           [BERHASIL] Tersimpan: {nama_bersih} ({len(biner)//1024} KB)")
                except Exception as e:
                    print(f"           [GAGAL] {e}")
                
                sb.sleep(random.uniform(INTERVAL_JEDA[0], INTERVAL_JEDA[1]))

print("=" * 70)


---
## 6. Konversi Dokumen: PDF ke Teks Mentah (Tahap 3)
Mengekstrak teks penuh dari seluruh berkas dokumen PDF sah menggunakan library `PyMuPDF` (`fitz`), melakukan pembersihan header berulang, dan menyimpannya sebagai berkas korpus `.txt`.


In [ ]:
# ======================================================================
# TUGAS 3: Web Crawler & Information Extraction Direktori Putusan MA RI
# Anggota: Akhmad Zamri Ardani (202310370311406) | M Rifqi Dzaky Arrafi (202310370311441)
# Informatika - Fakultas Teknik - Universitas Muhammadiyah Malang
# ======================================================================
# Sel 6: Konversi Teks Penuh dari PDF ke TXT (Tahap 3)

import os
import re
import sys
from pathlib import Path
import fitz  # PyMuPDF

if 'DIRECTORIES' not in globals():
    ENV_IS_COLAB = 'google.colab' in sys.modules
    BASE_PATH = Path('/content/drive/MyDrive/PN JEMBER/data') if ENV_IS_COLAB else Path('./data')
    DIRECTORIES = {
        'pdf': BASE_PATH / '02_pdf',
        'txt': BASE_PATH / '03_txt',
    }
    DIRECTORIES['txt'].mkdir(parents=True, exist_ok=True)

daftar_pdf = sorted([p for p in DIRECTORIES['pdf'].glob('*.pdf') if not p.name.startswith('._')])

print("=" * 70)
print("KONVERSI BERKAS DOKUMEN PDF KE TEKS MURNI (.TXT)")
print("=" * 70)
print(f"Total Dokumen PDF Terdeteksi : {len(daftar_pdf)} Berkas")
print(f"Direktori Target Output      : {DIRECTORIES['txt']}")
print("-" * 70)

sukses_konversi = 0
total_karakter  = 0

for idx, p in enumerate(daftar_pdf, 1):
    txt_out = DIRECTORIES['txt'] / f"{p.stem}.txt"
    try:
        doc = fitz.open(p)
        halaman_teks = []
        for hal in doc:
            t = hal.get_text('text')
            # Bersihkan watermark umum direktori MA RI
            t = re.sub(r'(?i)direktori\s+putusan\s+mahkamah\s+agung\s+republik\s+indonesia', '', t)
            t = re.sub(r'(?i)halaman\s+\d+\s+dari\s+\d+\s+hal\.\s+putusan\s+nomor\s+[^\n]+', '', t)
            halaman_teks.append(t)
        doc.close()
        
        teks_penuh = re.sub(r'\n{3,}', '\n\n', '\n'.join(halaman_teks)).strip()
        txt_out.write_text(teks_penuh, encoding='utf-8')
        sukses_konversi += 1
        total_karakter  += len(teks_penuh)
        
        if idx % 25 == 0 or idx == len(daftar_pdf):
            print(f"  [PROGRES] Berhasil mengonversi: {idx:>3d} / {len(daftar_pdf)} dokumen")
    except Exception as e:
        print(f"  [GAGAL] {p.name}: {e}")

print("-" * 70)
print(f"Total Berkas Berhasil Dikonversi : {sukses_konversi} Dokumen TXT")
print(f"Rata-rata Ukuran Teks per Berkas : {total_karakter // max(1, sukses_konversi):,} Karakter")
print("=" * 70)


---
## 7. Ekstraksi Informasi Putusan ke Format CSV (Tahap 4 - Model Regex)
Mengekstrak 4 entitas wajib (**Nomor Putusan**, **Lembaga Peradilan**, **Barang Bukti**, **Amar Putusan**) serta 2 entitas tambahan untuk nilai lebih (**Nama Terdakwa** dan **Tuntutan JPU**) berbasis Regular Expression ke dalam `hasil_ekstraksi.csv`.


In [ ]:
# ======================================================================
# TUGAS 3: Web Crawler & Information Extraction Direktori Putusan MA RI
# Anggota: Akhmad Zamri Ardani (202310370311406) | M Rifqi Dzaky Arrafi (202310370311441)
# Informatika - Fakultas Teknik - Universitas Muhammadiyah Malang
# ======================================================================
# Sel 7: Information Extraction Entitas Putusan dengan Model Regex (Tahap 4)

import os
import re
import sys
from pathlib import Path
import pandas as pd

if 'DIRECTORIES' not in globals():
    ENV_IS_COLAB = 'google.colab' in sys.modules
    BASE_PATH = Path('/content/drive/MyDrive/PN JEMBER/data') if ENV_IS_COLAB else Path('./data')
    DIRECTORIES = {
        'txt':   BASE_PATH / '03_txt',
        'hasil': BASE_PATH / '04_hasil',
    }
    DIRECTORIES['hasil'].mkdir(parents=True, exist_ok=True)

# 1. Ekstraksi Nomor Putusan (Wajib)
def ekstrak_no_putusan(teks):
    m = re.search(r'(?i)Nomor\s+(\d+[\w\.\/]+(?:Pid\.Sus|Pid\.B|Pid)[\w\.\/]*\d{4}[\w\.\/]*PN\s*(?:Jmr|Jember))', teks)
    if m:
        return f"Nomor {m.group(1).strip()}"
    m2 = re.search(r'(?i)Nomor\s+(\d+\s*[\/\.\w\-]+\s*\/\s*Pid[^\n]{5,60})', teks)
    if m2:
        return f"Nomor {re.sub(r'\s+', ' ', m2.group(1)).strip()}"
    return ''

# 2. Ekstraksi Lembaga Peradilan (Wajib)
def ekstrak_pengadilan(teks):
    m = re.search(r'(?i)(Pengadilan\s+Negeri\s+Jember)', teks)
    return m.group(1).strip() if m else 'Pengadilan Negeri Jember'

# 3. Ekstraksi Barang Bukti (Wajib)
def ekstrak_barang_bukti(teks):
    m = re.search(r'(?i)(?:barang bukti|menetapkan barang bukti).*?:?\s*\n([\s\S]*?)(?:\n\s*\d+\.\s*membeban|\n\s*membebankan|menimbang)', teks)
    if m:
        bb = m.group(1).strip()
        return re.sub(r'\s{2,}', ' ', bb)[:1500]
    return ''

# 4. Ekstraksi Amar Putusan (Wajib)
def ekstrak_amar_putusan(teks):
    m = re.search(r'(?i)(?:M E N G A D I L I|MENGADILI)\s*:?\s*\n([\s\S]*?)(?:Demikianlah|Demikian diputuskan|Panitera Pengganti)', teks)
    if m:
        amar = m.group(1).strip()
        return re.sub(r'\s{2,}', ' ', amar)[:2500]
    return ''

# 5. Fitur Nilai Lebih: Nama Terdakwa
def ekstrak_terdakwa(teks):
    m = re.search(r'(?i)(?:Nama\s+lengkap|Nama)\s*:\s*([^\n;]+)', teks)
    if m:
        return m.group(1).strip()
    m2 = re.search(r'(?i)terdakwa\s+([A-Z\s]{3,35})\s+(?:anak\s+dari|bin|binti)', teks)
    if m2:
        return m2.group(1).strip()
    return ''

# 6. Fitur Nilai Lebih: Tuntutan Penuntut Umum (JPU)
def ekstrak_tuntutan(teks):
    m = re.search(r'(?i)(?:M\s*E\s*N\s*U\s*N\s*T\s*U\s*T|MENUNTUT)\s*:?\s*\n([\s\S]*?)(?:Menimbang|Mengingat)', teks)
    if m:
        t = re.sub(r'\s{2,}', ' ', m.group(1).strip())
        return t[:1000]
    return ''

daftar_txt = sorted([t for t in DIRECTORIES['txt'].glob('*.txt') if not t.name.startswith('._')])

print("=" * 70)
print("EKSTRAKSI INFORMASI ENTITAS PUTUSAN (MODEL REGEX)")
print("=" * 70)
print(f"Total Dokumen Teks Dianalisis : {len(daftar_txt)} Berkas")
print("-" * 70)

data_ekstraksi = []
for idx, txt_path in enumerate(daftar_txt, 1):
    teks_dok   = txt_path.read_text(encoding='utf-8', errors='replace')
    no_put     = ekstrak_no_putusan(teks_dok)
    pengadilan = ekstrak_pengadilan(teks_dok)
    terdakwa   = ekstrak_terdakwa(teks_dok)
    bb         = ekstrak_barang_bukti(teks_dok)
    tuntutan   = ekstrak_tuntutan(teks_dok)
    amar       = ekstrak_amar_putusan(teks_dok)

    data_ekstraksi.append({
        'no': idx,
        'no_putusan': no_put,
        'lembaga_peradilan': pengadilan,
        'terdakwa': terdakwa,
        'barang_bukti': bb,
        'tuntutan': tuntutan,
        'amar_putusan': amar,
        'file': txt_path.name,
    })

FILE_HASIL_CSV = DIRECTORIES['hasil'] / 'hasil_ekstraksi.csv'
df_hasil = pd.DataFrame(data_ekstraksi)
df_hasil.to_csv(FILE_HASIL_CSV, index=False, encoding='utf-8-sig')

print(f"Penyimpanan File CSV : {FILE_HASIL_CSV}")
print(f"Total Baris Data     : {len(df_hasil)} Baris Putusan Terstruktur")
print("-" * 70)
print("[STATUS KELENGKAPAN ENTITAS TERESTRAKSI]:")
for col in ['no_putusan', 'lembaga_peradilan', 'terdakwa', 'barang_bukti', 'tuntutan', 'amar_putusan']:
    terisi = sum(1 for v in df_hasil[col] if str(v).strip() != '')
    pct = (terisi / len(df_hasil) * 100) if len(df_hasil) > 0 else 0
    print(f"  - Entitas '{col:<18s}': {terisi:>3d} / {len(df_hasil)} ({pct:.1f}%)")
print("=" * 70)


---
## 8. Audit Kualitas Data & Pembuatan Berkas Overview.xlsx (Tahap 5)
Mengevaluasi rasio kelengkapan entitas wajib, menampilkan tabel pratinjau data, serta mengekspor berkas rekapitulasi ke format `Overview.xlsx` sesuai standar pelaporan penugasan.


In [ ]:
# ======================================================================
# TUGAS 3: Web Crawler & Information Extraction Direktori Putusan MA RI
# Anggota: Akhmad Zamri Ardani (202310370311406) | M Rifqi Dzaky Arrafi (202310370311441)
# Informatika - Fakultas Teknik - Universitas Muhammadiyah Malang
# ======================================================================
# Sel 8: Validasi Kualitas Dataset, Pratinjau & Ekspor Overview.xlsx

import sys
from pathlib import Path
import pandas as pd

if 'DIRECTORIES' not in globals():
    ENV_IS_COLAB = 'google.colab' in sys.modules
    BASE_PATH = Path('/content/drive/MyDrive/PN JEMBER/data') if ENV_IS_COLAB else Path('./data')
    DIRECTORIES = {'hasil': BASE_PATH / '04_hasil'}

FILE_HASIL_CSV     = DIRECTORIES['hasil'] / 'hasil_ekstraksi.csv'
FILE_OVERVIEW_XLSX = DIRECTORIES['hasil'] / 'Overview.xlsx'

print("=" * 70)
print("AUDIT KUALITAS DATASET & REKAPITULASI LAPORAN OVERVIEW.XLSX")
print("=" * 70)

if not FILE_HASIL_CSV.exists():
    print("[PERINGATAN] Berkas hasil ekstraksi belum tersedia. Jalankan Sel 7 terlebih dahulu.")
else:
    df_final = pd.read_csv(FILE_HASIL_CSV, dtype=str).fillna('')
    print(f"Total Dokumen Terverifikasi : {len(df_final)} Putusan")
    print("-" * 70)
    print("Ringkasan Kelengkapan Kolom Wajib:")
    for col in ['no_putusan', 'lembaga_peradilan', 'barang_bukti', 'amar_putusan']:
        terisi = sum(1 for v in df_final[col] if str(v).strip() != '')
        pct = (terisi / len(df_final) * 100) if len(df_final) > 0 else 0
        print(f"  - Kolom '{col:<18s}': {terisi:>3d} / {len(df_final)} ({pct:.1f}%)")

    print("-" * 70)
    print("Pratinjau 5 Baris Pertama:")
    pd.set_option('display.max_colwidth', 40)
    pratinjau_cols = ['no', 'no_putusan', 'lembaga_peradilan', 'terdakwa']
    print(df_final[pratinjau_cols].head(5).to_string(index=False))
    print("-" * 70)

    try:
        cols_overview = ['no', 'no_putusan', 'lembaga_peradilan', 'barang_bukti', 'amar_putusan']
        df_final[cols_overview].to_excel(FILE_OVERVIEW_XLSX, index=False)
        print(f"[EKSPOR BERHASIL] Berkas Overview.xlsx tersimpan di:")
        print(f"  -> {FILE_OVERVIEW_XLSX.resolve()}")
    except Exception as e:
        print(f"[CATATAN] Ekspor xlsx dilewati: {e}")

    print("=" * 70)
    print("[SELESAI] Seluruh tahapan Tugas 3 berhasil dituntaskan.")
    print("Dataset siap digunakan sebagai input pemrosesan Tugas 4 (Indexing).")
    print("=" * 70)
